# DeNICE Matched Oracle / BestAllowedRoute

Attach the original 100-clients dataset **and the previous denice_tip_diagnostics.zip** (or its extracted contents). Enable Internet. The notebook downloads results (4).zip from the old Drive link. Frozen evaluation only: no training or TIP refit.

Predictions, task availability and sample identities are replayed from the prior experiment. Oracle labels are used only in diagnostic selection. Bounds apply only to the enumerated route actions on this frozen checkpoint and sampled panel.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, zipfile, shutil, gc
RESULTS_DRIVE_URL = "https://drive.google.com/file/d/1BEjP4iGJbPT0uFcHZ7WXX4oM_1vyvx0M/view?usp=sharing"
RESULTS_ZIP = Path('/kaggle/working/results_03b9b53_full.zip')
DATA_DIR = "" # auto-discover mounted original dataset
TASK, ROUND, MAX_SAMPLES, BATCH_SIZE = 5, 19, 50000, 512
OUT = Path('/kaggle/working/denice_matched_03b9b53')
OUT.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/kaggle/working/denice_matched_source')
if not SOURCE.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/khoilv2005/FL_IL_IDS.git',str(SOURCE)],
                   check=True, env=dict(os.environ,GIT_LFS_SKIP_SMUDGE='1'))
else:
    subprocess.run(['git','-C',str(SOURCE),'pull','--ff-only'],check=True)
source_commit = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
sys.path.insert(0,str(SOURCE))
print('Evaluation source:', source_commit)
# Attach the previous denice_tip_diagnostics.zip as a Kaggle dataset.
# Kaggle may expose it as an extracted directory; both formats are accepted.
DIAGNOSTIC_INPUT = ""
if not DIAGNOSTIC_INPUT:
    candidates = list(Path('/kaggle/input').rglob('denice_tip_diagnostics.zip'))
    candidates += [p.parent for p in Path('/kaggle/input').rglob('router_diagnostics.json')
                   if (p.parent/'predictions.csv').exists() and (p.parent/'profile_manifest.json').exists()]
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the prior TIP diagnostic output or set DIAGNOSTIC_INPUT. Candidates: {candidates}')
    DIAGNOSTIC_INPUT = str(candidates[0])
print('Prior frozen predictions:',DIAGNOSTIC_INPUT)


In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score
from fed_learning.data.incremental_loader import IncrementalDataLoader
from fed_learning.training.denice_delta_checkpoint import load_denice_checkpoint
from fed_learning.training.decentralized_denice_il import _limit_eval_samples, _partition_test_data_by_client
from fed_learning.training.denice_eval import _denice_routed_logits_with_episodes
from eval_checkpoint import _make_denice_client_model
if not DATA_DIR or not (Path(DATA_DIR, 'metadata.json').exists() and Path(DATA_DIR, 'global_test_data.npz').exists()):
    input_root = Path('/kaggle/input')
    candidates = sorted({p.parent for p in input_root.rglob('metadata.json')
                         if (p.parent/'global_test_data.npz').exists()})
    historical = Path('/kaggle/input/datasets/khoilv2005/100-clients/100-clients')
    if historical in candidates:
        DATA_DIR = str(historical)
    elif len(candidates) == 1:
        DATA_DIR = str(candidates[0])
    else:
        raise FileNotFoundError(f'Could not uniquely find original dataset under {input_root}. Candidates: {candidates}. Attach the 100-clients dataset or set DATA_DIR.')
print('Using dataset:', DATA_DIR)
if 'PASTE_GOOGLE_DRIVE' in RESULTS_DRIVE_URL:
    raise ValueError('Paste the Google Drive share URL for the full-run ZIP in RESULTS_DRIVE_URL.')
if not RESULTS_ZIP.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'], check=True)
    subprocess.run([sys.executable, '-m', 'gdown', '--fuzzy', RESULTS_DRIVE_URL, '-O', str(RESULTS_ZIP)], check=True)
if not RESULTS_ZIP.exists() or RESULTS_ZIP.stat().st_size == 0:
    raise FileNotFoundError(f'Google Drive download failed: {RESULTS_ZIP}')
name = f'checkpoint_task_{TASK}_round_{ROUND}.pt'
root = OUT/'checkpoints'
root.mkdir(exist_ok=True)
with zipfile.ZipFile(RESULTS_ZIP) as z:
    matches = [n for n in z.namelist() if Path(n).name == name]
    if len(matches) != 1: raise ValueError(f'Expected one {name}; found {len(matches)}')
    prefix = matches[0][:-len(name)]
    for n in z.namelist():
        if not n.startswith(prefix) or n.endswith('/'): continue
        base = Path(n).name
        if base in ('config.json','final_report.json','checkpoint_index.json') or base == f'checkpoint_task_{TASK}_base.pt' or base.startswith(f'checkpoint_task_{TASK}_round_'):
            with z.open(n) as src, (root/base).open('wb') as dst: shutil.copyfileobj(src,dst)
config = json.loads((root/'config.json').read_text())
report = json.loads((root/'final_report.json').read_text())
if not str(config.get('git_commit','')).startswith('03b9b53'):
    raise ValueError('Expected baseline full run, training commit 03b9b53')
if report['completed_task'] != TASK: raise ValueError('Report must match final task')
ckpt = load_denice_checkpoint(str(root/name))
ids = [int(cid) for cid in report['final_metrics']['per_client']]
if not ids or not set(ids) <= {int(cid) for cid in ckpt['client_model_states']}:
    raise ValueError('Missing original evaluation clients')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training source:',config['git_commit'],'clients:',len(ids),'device:',device)


In [ ]:
from tools.eval_denice_tip_router import run_suite, write_json
from threadpoolctl import threadpool_limits
import hashlib
loader = IncrementalDataLoader(DATA_DIR)
classes = {t:list(map(int,loader.get_task_classes(t))) for t in range(TASK+1)}
seed = int(config.get('random_seed',config.get('seed',42)))
x,y = loader.get_test_data(TASK,cumulative=True)
# Carry original test-row identities through exactly the existing samplers.
indices, selected_y, sample_info = _limit_eval_samples(
    torch.arange(len(y))[:,None], y, MAX_SAMPLES, seed+TASK)
index_shards, partition = _partition_test_data_by_client(
    indices, selected_y, ids, seed+104729*TASK)
shards = {}
for cid, shard in index_shards.items():
    selected = shard['X_test'].reshape(-1).long()
    shards[cid] = dict(X_test=x[selected],y_test=shard['y_test'],sample_ids=selected.numpy())
loader._test_data = None
del x,y,indices,index_shards
gc.collect()
digest = hashlib.sha256()
with (root/name).open('rb') as f:
    for block in iter(lambda:f.read(8*1024*1024),b''):
        digest.update(block)
protocol = dict(
    kind='matched oracle and best allowed route; no router fitting',
    training_commit=config['git_commit'],evaluation_commit=source_commit,
    checkpoint=name,checkpoint_file_sha256=digest.hexdigest(),
    data_dir=DATA_DIR,diagnostic_input=DIAGNOSTIC_INPUT,seed=seed,
    task_classes=classes,client_ids=ids,sample_info=sample_info,partition=partition)
write_json(OUT/'protocol.json',protocol)


In [ ]:
from tools.eval_denice_matched_oracle import run_matched_oracle
try:
    with threadpool_limits(limits=1):
        summary = run_matched_oracle(ckpt,shards,classes,ids,OUT,device,
                                     DIAGNOSTIC_INPUT,batch_size=BATCH_SIZE)
    print(summary.to_string(index=False))
    display(summary)
finally:
    archive = Path('/kaggle/working/denice_matched_oracle_diagnostics.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob('*'):
            if p.is_file() and 'checkpoints' not in p.relative_to(OUT).parts:
                z.write(p,str(p.relative_to(OUT)))
    print('Diagnostic output (may be partial if execution failed):',archive)
